In [1]:
"""
=============================================================================
 TRANSFORMER FROM SCRATCH — Complete Implementation in PyTorch
=============================================================================
 Every component (Scaled Dot-Product Attention, Multi-Head Attention,
 Positional Encoding, Feed-Forward Network, Encoder, Decoder) is built
 from first principles. Trained on a small synthetic EN→FR dataset so
 the script is fully self-contained — no external downloads needed.
"""

import math
import copy
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter

# ── reproducibility ─────────────────────────────────────────────────────────
torch.manual_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ============================================================================
# 1.  SCALED DOT-PRODUCT ATTENTION
# ============================================================================
#   Attention(Q, K, V) = softmax(Q·Kᵀ / √d_k) · V
#
#   • Q, K, V  — query, key, value matrices
#   • d_k      — dimension of keys (used to scale the dot products)
#   • mask     — optional mask to prevent attending to certain positions
# ============================================================================

def scaled_dot_product_attention(query, key, value, mask=None, dropout=None):
    """
    Args
    ----
    query : (batch, heads, seq_len_q, d_k)
    key   : (batch, heads, seq_len_k, d_k)
    value : (batch, heads, seq_len_v, d_v)   [seq_len_k == seq_len_v]
    mask  : broadcastable to (batch, 1, seq_len_q, seq_len_k)

    Returns
    -------
    output          : (batch, heads, seq_len_q, d_v)
    attention_weights: (batch, heads, seq_len_q, seq_len_k)
    """
    d_k = query.size(-1)

    # Step 1 — dot product between Q and K^T
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)

    # Step 2 — apply mask (for padding or causal / look-ahead masking)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-1e9"))

    # Step 3 — softmax to get attention weights
    attention_weights = F.softmax(scores, dim=-1)

    if dropout is not None:
        attention_weights = dropout(attention_weights)

    # Step 4 — weighted sum of values
    output = torch.matmul(attention_weights, value)
    return output, attention_weights


# ============================================================================
# 2.  MULTI-HEAD ATTENTION
# ============================================================================
#   Instead of one attention function, we project Q, K, V  h  times with
#   different learned linear projections, run attention in parallel, then
#   concatenate and project again.
# ============================================================================

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model   = d_model
        self.num_heads = num_heads
        self.d_k       = d_model // num_heads   # dimension per head

        # Linear projections for Q, K, V and output
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        # 1) Linear projections  →  (batch, heads, seq_len, d_k)
        Q = self.W_q(query).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        # 2) Apply attention
        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask, self.dropout)

        # 3) Concatenate heads  →  (batch, seq_len, d_model)
        attn_output = (
            attn_output.transpose(1, 2)
            .contiguous()
            .view(batch_size, -1, self.d_model)
        )

        # 4) Final linear projection
        return self.W_o(attn_output)


# ============================================================================
# 3.  POSITION-WISE FEED-FORWARD NETWORK
# ============================================================================
#   FFN(x) = ReLU(x·W₁ + b₁)·W₂ + b₂
#   Applied independently to each position.
# ============================================================================

class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.fc2(self.dropout(F.relu(self.fc1(x))))


# ============================================================================
# 4.  POSITIONAL ENCODING  (sinusoidal — no learnable parameters)
# ============================================================================
#   PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
#   PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
# ============================================================================

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)                       # (max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float() # (max_len, 1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)   # even indices
        pe[:, 1::2] = torch.cos(position * div_term)   # odd  indices

        pe = pe.unsqueeze(0)  # (1, max_len, d_model) for broadcasting
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, : x.size(1), :]
        return self.dropout(x)


# ============================================================================
# 5.  ENCODER LAYER  &  ENCODER STACK
# ============================================================================

class EncoderLayer(nn.Module):
    """One layer: Self-Attention → Add & Norm → FFN → Add & Norm"""

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn  = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn        = PositionWiseFeedForward(d_model, d_ff, dropout)
        self.norm1      = nn.LayerNorm(d_model)
        self.norm2      = nn.LayerNorm(d_model)
        self.dropout1   = nn.Dropout(dropout)
        self.dropout2   = nn.Dropout(dropout)

    def forward(self, src, src_mask=None):
        # Sub-layer 1: multi-head self-attention
        attn_out = self.self_attn(src, src, src, src_mask)
        src = self.norm1(src + self.dropout1(attn_out))

        # Sub-layer 2: feed-forward
        ffn_out = self.ffn(src)
        src = self.norm2(src + self.dropout2(ffn_out))
        return src


class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_enc   = PositionalEncoding(d_model, dropout=dropout)
        self.layers    = nn.ModuleList(
            [EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.d_model = d_model

    def forward(self, src, src_mask=None):
        x = self.embedding(src) * math.sqrt(self.d_model)
        x = self.pos_enc(x)
        for layer in self.layers:
            x = layer(x, src_mask)
        return x


# ============================================================================
# 6.  DECODER LAYER  &  DECODER STACK
# ============================================================================

class DecoderLayer(nn.Module):
    """Masked Self-Attn → Add&Norm → Cross-Attn → Add&Norm → FFN → Add&Norm"""

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn   = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn  = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn         = PositionWiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, tgt, enc_output, src_mask=None, tgt_mask=None):
        # Sub-layer 1: masked self-attention (causal)
        attn_out = self.self_attn(tgt, tgt, tgt, tgt_mask)
        tgt = self.norm1(tgt + self.dropout1(attn_out))

        # Sub-layer 2: cross-attention over encoder output
        attn_out = self.cross_attn(tgt, enc_output, enc_output, src_mask)
        tgt = self.norm2(tgt + self.dropout2(attn_out))

        # Sub-layer 3: feed-forward
        ffn_out = self.ffn(tgt)
        tgt = self.norm3(tgt + self.dropout3(ffn_out))
        return tgt


class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_enc   = PositionalEncoding(d_model, dropout=dropout)
        self.layers    = nn.ModuleList(
            [DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.d_model = d_model

    def forward(self, tgt, enc_output, src_mask=None, tgt_mask=None):
        x = self.embedding(tgt) * math.sqrt(self.d_model)
        x = self.pos_enc(x)
        for layer in self.layers:
            x = layer(x, enc_output, src_mask, tgt_mask)
        return x


# ============================================================================
# 7.  FULL TRANSFORMER  (Encoder + Decoder + Output Projection)
# ============================================================================

class Transformer(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model=256,
        num_heads=8,
        d_ff=512,
        num_layers=3,
        dropout=0.1,
        max_len=100,
    ):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, d_model, num_heads, d_ff, num_layers, dropout)
        self.decoder = Decoder(tgt_vocab_size, d_model, num_heads, d_ff, num_layers, dropout)
        self.output_projection = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        enc_output = self.encoder(src, src_mask)
        dec_output = self.decoder(tgt, enc_output, src_mask, tgt_mask)
        logits     = self.output_projection(dec_output)
        return logits

    # ── mask utilities ──────────────────────────────────────────────────────
    @staticmethod
    def make_src_mask(src, pad_idx):
        """(batch, 1, 1, src_len) — masks <PAD> tokens"""
        return (src != pad_idx).unsqueeze(1).unsqueeze(2)

    @staticmethod
    def make_tgt_mask(tgt, pad_idx):
        """Combines padding mask with causal (look-ahead) mask."""
        batch, tgt_len = tgt.shape
        pad_mask = (tgt != pad_idx).unsqueeze(1).unsqueeze(2)          # (B,1,1,T)
        causal   = torch.tril(torch.ones(tgt_len, tgt_len, device=tgt.device)).bool()  # (T,T)
        return pad_mask & causal  # (B,1,T,T) via broadcast


# ============================================================================
# 8.  VOCABULARY & TOKENIZER  (simple word-level)
# ============================================================================

class SimpleVocab:
    PAD_TOKEN = "<PAD>"
    SOS_TOKEN = "<SOS>"
    EOS_TOKEN = "<EOS>"
    UNK_TOKEN = "<UNK>"
    SPECIAL   = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN]

    def __init__(self, sentences, min_freq=1):
        counter = Counter()
        for s in sentences:
            counter.update(s.lower().split())
        self.itos = list(self.SPECIAL) + [w for w, c in counter.items() if c >= min_freq]
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def encode(self, sentence):
        tokens = [self.stoi.get(w, self.stoi[self.UNK_TOKEN]) for w in sentence.lower().split()]
        return [self.stoi[self.SOS_TOKEN]] + tokens + [self.stoi[self.EOS_TOKEN]]

    def decode(self, indices):
        return " ".join(
            self.itos[i] for i in indices
            if self.itos[i] not in {self.PAD_TOKEN, self.SOS_TOKEN, self.EOS_TOKEN}
        )

    @property
    def pad_idx(self):
        return self.stoi[self.PAD_TOKEN]

    def __len__(self):
        return len(self.itos)


# ============================================================================
# 9.  SYNTHETIC EN→FR DATASET  (self-contained, no downloads)
# ============================================================================

PARALLEL_DATA = [
    ("the cat sits on the mat",           "le chat est assis sur le tapis"),
    ("i love this movie",                 "j'aime ce film"),
    ("she is reading a book",             "elle lit un livre"),
    ("they are playing in the park",      "ils jouent dans le parc"),
    ("he runs every morning",             "il court chaque matin"),
    ("we eat breakfast together",         "nous prenons le petit déjeuner ensemble"),
    ("the dog is sleeping",              "le chien dort"),
    ("she writes beautiful poems",        "elle écrit de beaux poèmes"),
    ("the sun is shining brightly",       "le soleil brille intensément"),
    ("i am learning french",              "j'apprends le français"),
    ("he plays the guitar well",          "il joue bien de la guitare"),
    ("the children laugh happily",        "les enfants rient joyeusement"),
    ("we travel to paris",                "nous voyageons à paris"),
    ("they study mathematics",            "ils étudient les mathématiques"),
    ("the flowers bloom in spring",       "les fleurs fleurissent au printemps"),
    ("she drinks coffee every day",       "elle boit du café chaque jour"),
    ("i watch the stars at night",        "je regarde les étoiles la nuit"),
    ("the bird sings a song",             "l'oiseau chante une chanson"),
    ("he reads the newspaper",            "il lit le journal"),
    ("we walk along the river",           "nous marchons le long de la rivière"),
    ("the teacher explains the lesson",   "le professeur explique la leçon"),
    ("she paints a beautiful picture",    "elle peint un beau tableau"),
    ("they cook dinner together",         "ils préparent le dîner ensemble"),
    ("the rain falls softly",             "la pluie tombe doucement"),
    ("i write a letter to my friend",     "j'écris une lettre à mon ami"),
    ("he swims in the ocean",             "il nage dans l'océan"),
    ("we listen to music",                "nous écoutons de la musique"),
    ("the baby sleeps peacefully",        "le bébé dort paisiblement"),
    ("she dances with grace",             "elle danse avec grâce"),
    ("they build a sandcastle",           "ils construisent un château de sable"),
]


class TranslationDataset(Dataset):
    def __init__(self, pairs, src_vocab, tgt_vocab, max_len=20):
        self.pairs     = pairs
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.max_len   = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src_sent, tgt_sent = self.pairs[idx]
        src_ids = self.src_vocab.encode(src_sent)[: self.max_len]
        tgt_ids = self.tgt_vocab.encode(tgt_sent)[: self.max_len]
        return torch.tensor(src_ids), torch.tensor(tgt_ids)


def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_padded = nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=0)
    tgt_padded = nn.utils.rnn.pad_sequence(tgt_batch, batch_first=True, padding_value=0)
    return src_padded, tgt_padded


# ============================================================================
# 10.  TRAINING LOOP
# ============================================================================

def train_epoch(model, dataloader, optimizer, criterion, src_vocab, tgt_vocab):
    model.train()
    total_loss = 0
    for src, tgt in dataloader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)

        tgt_input  = tgt[:, :-1]   # teacher forcing: input  = <SOS> ... t_{n-1}
        tgt_output = tgt[:, 1:]    # expected output           = t_1  ... <EOS>

        src_mask = Transformer.make_src_mask(src, src_vocab.pad_idx).to(DEVICE)
        tgt_mask = Transformer.make_tgt_mask(tgt_input, tgt_vocab.pad_idx).to(DEVICE)

        logits = model(src, tgt_input, src_mask, tgt_mask)  # (B, T, V)
        logits = logits.reshape(-1, logits.size(-1))
        tgt_output = tgt_output.reshape(-1)

        loss = criterion(logits, tgt_output)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(dataloader)


# ============================================================================
# 11.  GREEDY DECODE  (inference)
# ============================================================================

@torch.no_grad()
def greedy_decode(model, src_sentence, src_vocab, tgt_vocab, max_len=20):
    model.eval()
    src_ids  = torch.tensor([src_vocab.encode(src_sentence)]).to(DEVICE)
    src_mask = Transformer.make_src_mask(src_ids, src_vocab.pad_idx).to(DEVICE)
    enc_out  = model.encoder(src_ids, src_mask)

    tgt_ids = [tgt_vocab.stoi[SimpleVocab.SOS_TOKEN]]

    for _ in range(max_len):
        tgt_tensor = torch.tensor([tgt_ids]).to(DEVICE)
        tgt_mask   = Transformer.make_tgt_mask(tgt_tensor, tgt_vocab.pad_idx).to(DEVICE)
        dec_out    = model.decoder(tgt_tensor, enc_out, src_mask, tgt_mask)
        logits     = model.output_projection(dec_out[:, -1, :])
        next_token = logits.argmax(dim=-1).item()
        tgt_ids.append(next_token)
        if next_token == tgt_vocab.stoi[SimpleVocab.EOS_TOKEN]:
            break

    return tgt_vocab.decode(tgt_ids)


# ============================================================================
# 12.  MAIN — BUILD, TRAIN, EVALUATE
# ============================================================================

def main():
    print("=" * 70)
    print("  TRANSFORMER FROM SCRATCH — EN → FR Translation Demo")
    print("=" * 70)

    # ── build vocabs ────────────────────────────────────────────────────────
    src_sents = [p[0] for p in PARALLEL_DATA]
    tgt_sents = [p[1] for p in PARALLEL_DATA]
    src_vocab = SimpleVocab(src_sents)
    tgt_vocab = SimpleVocab(tgt_sents)
    print(f"\nSource vocab size : {len(src_vocab)}")
    print(f"Target vocab size : {len(tgt_vocab)}")

    # ── dataset & dataloader ────────────────────────────────────────────────
    dataset    = TranslationDataset(PARALLEL_DATA, src_vocab, tgt_vocab)
    dataloader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)

    # ── model ───────────────────────────────────────────────────────────────
    D_MODEL    = 128
    NUM_HEADS  = 4
    D_FF       = 256
    NUM_LAYERS = 2
    DROPOUT    = 0.1

    model = Transformer(
        src_vocab_size=len(src_vocab),
        tgt_vocab_size=len(tgt_vocab),
        d_model=D_MODEL,
        num_heads=NUM_HEADS,
        d_ff=D_FF,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
    ).to(DEVICE)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters  : {total_params:,}")

    # ── optimizer & loss ────────────────────────────────────────────────────
    optimizer = optim.Adam(model.parameters(), lr=3e-4, betas=(0.9, 0.98), eps=1e-9)
    criterion = nn.CrossEntropyLoss(ignore_index=tgt_vocab.pad_idx)

    # ── training ────────────────────────────────────────────────────────────
    NUM_EPOCHS = 80
    print(f"\nTraining for {NUM_EPOCHS} epochs ...\n")
    t0 = time.time()

    for epoch in range(1, NUM_EPOCHS + 1):
        loss = train_epoch(model, dataloader, optimizer, criterion, src_vocab, tgt_vocab)
        if epoch % 10 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{NUM_EPOCHS}  |  Loss: {loss:.4f}")

    elapsed = time.time() - t0
    print(f"\nTraining completed in {elapsed:.1f}s")

    # ── inference demo ──────────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("  TRANSLATION RESULTS  (greedy decoding)")
    print("=" * 70 + "\n")

    test_sentences = [
        "the cat sits on the mat",
        "i love this movie",
        "she is reading a book",
        "the sun is shining brightly",
        "we walk along the river",
        "the bird sings a song",
    ]

    for sent in test_sentences:
        translation = greedy_decode(model, sent, src_vocab, tgt_vocab)
        print(f"  EN: {sent}")
        print(f"  FR: {translation}\n")

    # ── architecture summary ────────────────────────────────────────────────
    print("=" * 70)
    print("  ARCHITECTURE SUMMARY")
    print("=" * 70)
    print(f"""
    ┌─────────────────────────────────────────────────────┐
    │                  TRANSFORMER                        │
    │                                                     │
    │  ┌──────────────────┐   ┌──────────────────┐       │
    │  │     ENCODER      │   │     DECODER      │       │
    │  │  (x{NUM_LAYERS} layers)     │   │  (x{NUM_LAYERS} layers)     │       │
    │  │                  │   │                  │       │
    │  │ ┌──────────────┐ │   │ ┌──────────────┐ │       │
    │  │ │  Self-Attn   │ │   │ │ Masked Self  │ │       │
    │  │ │  ({NUM_HEADS} heads)   │ │   │ │  Attention   │ │       │
    │  │ │  d_k={D_MODEL // NUM_HEADS}       │ │   │ │  ({NUM_HEADS} heads)   │ │       │
    │  │ └──────────────┘ │   │ └──────────────┘ │       │
    │  │ ┌──────────────┐ │   │ ┌──────────────┐ │       │
    │  │ │ Add & Norm   │ │   │ │ Cross-Attn   │ │       │
    │  │ └──────────────┘ │   │ │  ({NUM_HEADS} heads)   │ │       │
    │  │ ┌──────────────┐ │   │ └──────────────┘ │       │
    │  │ │   FFN        │ │   │ ┌──────────────┐ │       │
    │  │ │ {D_MODEL}→{D_FF}→{D_MODEL} │ │   │ │   FFN        │ │       │
    │  │ └──────────────┘ │   │ │ {D_MODEL}→{D_FF}→{D_MODEL} │ │       │
    │  │ ┌──────────────┐ │   │ └──────────────┘ │       │
    │  │ │ Add & Norm   │ │   │ ┌──────────────┐ │       │
    │  │ └──────────────┘ │   │ │ Linear→Vocab │ │       │
    │  └──────────────────┘   │ └──────────────┘ │       │
    │                         └──────────────────┘       │
    │                                                     │
    │  Positional Encoding: Sinusoidal (no learnable)    │
    │  Total Parameters  : {total_params:,}                  │
    └─────────────────────────────────────────────────────┘
    """)


if __name__ == "__main__":
    main()

Using device: cpu
  TRANSFORMER FROM SCRATCH — EN → FR Translation Demo

Source vocab size : 97
Target vocab size : 101
Model parameters  : 700,901

Training for 80 epochs ...

  Epoch   1/80  |  Loss: 4.6869
  Epoch  10/80  |  Loss: 3.0175
  Epoch  20/80  |  Loss: 1.8353
  Epoch  30/80  |  Loss: 0.9912
  Epoch  40/80  |  Loss: 0.5492
  Epoch  50/80  |  Loss: 0.2675
  Epoch  60/80  |  Loss: 0.1490
  Epoch  70/80  |  Loss: 0.0822
  Epoch  80/80  |  Loss: 0.0543

Training completed in 11.8s

  TRANSLATION RESULTS  (greedy decoding)

  EN: the cat sits on the mat
  FR: le chat est assis sur le tapis

  EN: i love this movie
  FR: j'aime ce film

  EN: she is reading a book
  FR: elle lit un livre

  EN: the sun is shining brightly
  FR: le soleil brille intensément

  EN: we walk along the river
  FR: nous marchons le long de la rivière

  EN: the bird sings a song
  FR: l'oiseau chante une chanson

  ARCHITECTURE SUMMARY

    ┌─────────────────────────────────────────────────────┐
    │ 

In [2]:
"""
=============================================================================
 TEXT SUMMARIZATION USING TRANSFORMERS — From Scratch (No Downloads)
=============================================================================
 Two complete approaches, both fully self-contained:
   1. ABSTRACTIVE Summarization — Encoder-Decoder Transformer that generates
      new summary text (trained on article→summary pairs)
   2. EXTRACTIVE Summarization — Encoder-only Transformer that scores and
      selects the most important sentences using attention weights

 Everything is built from raw PyTorch — no HuggingFace, no pre-trained models.
=============================================================================
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import re
import textwrap
from collections import Counter

torch.manual_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ============================================================================
# SHARED COMPONENTS: Transformer Building Blocks
# ============================================================================

class MultiHeadAttention(nn.Module):
    """Multi-head scaled dot-product attention."""
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        B = query.size(0)
        Q = self.W_q(query).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attn_weights = self.dropout(F.softmax(scores, dim=-1))
        out = torch.matmul(attn_weights, V)
        out = out.transpose(1, 2).contiguous().view(B, -1, self.num_heads * self.d_k)
        return self.W_o(out), attn_weights


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
    def forward(self, x):
        return self.net(x)


class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=2000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


# ============================================================================
# WORD-LEVEL VOCABULARY
# ============================================================================

class Vocab:
    SPECIALS = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]

    def __init__(self, sentences, min_freq=1):
        c = Counter()
        for s in sentences:
            c.update(self._tokenize(s))
        self.itos = list(self.SPECIALS) + [w for w, n in c.items() if n >= min_freq]
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def _tokenize(self, text):
        return re.findall(r"\b\w+\b", text.lower())

    def encode(self, text, add_special=True):
        ids = [self.stoi.get(w, self.stoi["<UNK>"]) for w in self._tokenize(text)]
        if add_special:
            ids = [self.stoi["<SOS>"]] + ids + [self.stoi["<EOS>"]]
        return ids

    def decode(self, ids):
        skip = {"<PAD>", "<SOS>", "<EOS>"}
        return " ".join(self.itos[i] for i in ids if self.itos[i] not in skip)

    @property
    def pad(self): return self.stoi["<PAD>"]
    def __len__(self): return len(self.itos)


# ============================================================================
# APPROACH 1: ABSTRACTIVE SUMMARIZATION (Encoder-Decoder Transformer)
# ============================================================================

class EncoderLayer(nn.Module):
    def __init__(self, d_model, nhead, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, nhead, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop1 = nn.Dropout(dropout)
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_out, attn_w = self.attn(x, x, x, mask)
        x = self.norm1(x + self.drop1(attn_out))
        x = self.norm2(x + self.drop2(self.ffn(x)))
        return x, attn_w


class DecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, nhead, dropout)
        self.cross_attn = MultiHeadAttention(d_model, nhead, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.drop1 = nn.Dropout(dropout)
        self.drop2 = nn.Dropout(dropout)
        self.drop3 = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask=None, tgt_mask=None):
        a, _ = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.drop1(a))
        a, cross_w = self.cross_attn(x, enc_out, enc_out, src_mask)
        x = self.norm2(x + self.drop2(a))
        x = self.norm3(x + self.drop3(self.ffn(x)))
        return x, cross_w


class AbstractiveSummarizer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=128, nhead=4, d_ff=256,
                 n_enc=2, n_dec=2, dropout=0.1):
        super().__init__()
        self.src_emb = nn.Embedding(len(src_vocab), d_model)
        self.tgt_emb = nn.Embedding(len(tgt_vocab), d_model)
        self.pos_enc = SinusoidalPE(d_model, dropout=dropout)
        self.enc_layers = nn.ModuleList([EncoderLayer(d_model, nhead, d_ff, dropout) for _ in range(n_enc)])
        self.dec_layers = nn.ModuleList([DecoderLayer(d_model, nhead, d_ff, dropout) for _ in range(n_dec)])
        self.out_proj = nn.Linear(d_model, len(tgt_vocab))
        self.d_model = d_model
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab

    def encode(self, src, src_mask=None):
        x = self.pos_enc(self.src_emb(src) * math.sqrt(self.d_model))
        all_attn = []
        for layer in self.enc_layers:
            x, attn_w = layer(x, src_mask)
            all_attn.append(attn_w)
        return x, all_attn

    def decode(self, tgt, enc_out, src_mask=None, tgt_mask=None):
        x = self.pos_enc(self.tgt_emb(tgt) * math.sqrt(self.d_model))
        cross_attns = []
        for layer in self.dec_layers:
            x, cw = layer(x, enc_out, src_mask, tgt_mask)
            cross_attns.append(cw)
        return self.out_proj(x), cross_attns

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        enc_out, enc_attn = self.encode(src, src_mask)
        logits, cross_attn = self.decode(tgt, enc_out, src_mask, tgt_mask)
        return logits, enc_attn, cross_attn

    def make_src_mask(self, src):
        return (src != self.src_vocab.pad).unsqueeze(1).unsqueeze(2)

    def make_tgt_mask(self, tgt):
        B, T = tgt.shape
        pad = (tgt != self.tgt_vocab.pad).unsqueeze(1).unsqueeze(2)
        causal = torch.tril(torch.ones(T, T, device=tgt.device)).bool()
        return pad & causal

    @torch.no_grad()
    def summarize(self, text, max_len=30):
        self.eval()
        src_ids = torch.tensor([self.src_vocab.encode(text)]).to(DEVICE)
        src_mask = self.make_src_mask(src_ids)
        enc_out, enc_attn = self.encode(src_ids, src_mask)

        tgt_ids = [self.tgt_vocab.stoi["<SOS>"]]
        for _ in range(max_len):
            tgt_t = torch.tensor([tgt_ids]).to(DEVICE)
            tgt_mask = self.make_tgt_mask(tgt_t)
            logits, cross_attn = self.decode(tgt_t, enc_out, src_mask, tgt_mask)
            next_tok = logits[0, -1].argmax().item()
            tgt_ids.append(next_tok)
            if next_tok == self.tgt_vocab.stoi["<EOS>"]:
                break
        return self.tgt_vocab.decode(tgt_ids), enc_attn, cross_attn


# ── Training data ───────────────────────────────────────────────────────────

SUMMARIZATION_DATA = [
    ("artificial intelligence has transformed many industries including healthcare finance and education "
     "researchers have developed large language models that can understand and generate human language "
     "these models are trained on massive datasets using powerful gpu clusters",
     "ai transforms industries through large language models trained on big data"),

    ("the global climate crisis continues to worsen as temperatures rise worldwide "
     "scientists warn that carbon emissions must be reduced dramatically to avoid catastrophic consequences "
     "renewable energy sources like solar and wind are becoming more affordable and widespread",
     "climate crisis worsens but renewable energy offers hope for reducing emissions"),

    ("quantum computers use qubits that can exist in multiple states simultaneously "
     "this allows them to solve certain problems much faster than classical computers "
     "major companies are investing billions in quantum computing research and development",
     "quantum computers with qubits solve problems faster than classical machines"),

    ("the stock market experienced significant volatility this quarter "
     "technology stocks led the decline while energy companies performed well "
     "analysts predict continued uncertainty due to inflation and interest rate concerns",
     "stock market volatile with tech declining and energy rising amid inflation worries"),

    ("electric vehicles are gaining popularity worldwide as battery costs decrease "
     "many countries have announced plans to ban new gasoline car sales by twenty thirty five "
     "charging infrastructure is expanding rapidly in urban and suburban areas",
     "electric vehicles grow popular as batteries cheapen and charging networks expand"),

    ("researchers discovered a new species of deep sea fish near hydrothermal vents "
     "the fish can survive extreme temperatures and high pressure environments "
     "this discovery provides insights into how life adapts to harsh conditions",
     "new deep sea fish species found near vents survives extreme conditions"),

    ("the world health organization reported a significant decrease in malaria cases globally "
     "new vaccines and preventive treatments have contributed to this decline "
     "however funding shortfalls threaten to reverse recent progress in affected regions",
     "malaria cases drop globally due to vaccines but funding gaps risk reversal"),

    ("space agencies are planning missions to mars within the next decade "
     "private companies like spacex are developing reusable rockets to reduce launch costs "
     "scientists are studying how long duration space travel affects human health",
     "mars missions planned as spacex cuts costs and scientists study space health effects"),

    ("the education sector is rapidly adopting online learning platforms "
     "students can now access courses from top universities around the world "
     "artificial intelligence tutors provide personalized learning experiences for each student",
     "education shifts to online platforms with ai providing personalized tutoring"),

    ("global semiconductor shortage continues to affect automobile production "
     "major car manufacturers have reduced output due to lack of computer chips "
     "governments are investing in domestic chip manufacturing to prevent future shortages",
     "chip shortage cuts car production as governments invest in domestic manufacturing"),

    ("cybersecurity threats are increasing as more businesses move operations online "
     "ransomware attacks have targeted hospitals schools and government agencies "
     "experts recommend multi factor authentication and regular security audits",
     "cybersecurity threats rise with ransomware hitting critical institutions"),

    ("the united nations climate conference brought together leaders from nearly two hundred countries "
     "they discussed ambitious targets for reducing greenhouse gas emissions "
     "developing nations called for greater financial support to transition to clean energy",
     "un climate conference sets emission targets as developing nations seek funding"),
]


def run_abstractive():
    print("\n" + "=" * 76)
    print("  APPROACH 1: ABSTRACTIVE SUMMARIZATION (Encoder-Decoder Transformer)")
    print("=" * 76)
    print("  The model GENERATES new summary text word by word.")

    articles = [p[0] for p in SUMMARIZATION_DATA]
    summaries = [p[1] for p in SUMMARIZATION_DATA]
    src_vocab = Vocab(articles)
    tgt_vocab = Vocab(summaries)
    print(f"\n  Source vocab: {len(src_vocab)} | Target vocab: {len(tgt_vocab)} | Pairs: {len(SUMMARIZATION_DATA)}")

    def pad_batch(encoded_list, pad_val):
        max_len = max(len(s) for s in encoded_list)
        return torch.tensor([s + [pad_val] * (max_len - len(s)) for s in encoded_list])

    src_enc = [src_vocab.encode(a) for a in articles]
    tgt_enc = [tgt_vocab.encode(s) for s in summaries]
    src_batch = pad_batch(src_enc, src_vocab.pad).to(DEVICE)
    tgt_batch = pad_batch(tgt_enc, tgt_vocab.pad).to(DEVICE)

    model = AbstractiveSummarizer(src_vocab, tgt_vocab).to(DEVICE)
    print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
    criterion = nn.CrossEntropyLoss(ignore_index=tgt_vocab.pad)

    EPOCHS = 150
    print(f"\n  Training {EPOCHS} epochs...")
    t0 = time.time()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        tgt_in, tgt_out = tgt_batch[:, :-1], tgt_batch[:, 1:]
        logits, _, _ = model(src_batch, tgt_in, model.make_src_mask(src_batch), model.make_tgt_mask(tgt_in))
        loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if epoch % 30 == 0 or epoch == 1:
            print(f"    Epoch {epoch:3d}  Loss: {loss.item():.4f}")
    print(f"  Done in {time.time()-t0:.1f}s")

    # Inference
    print(f"\n  {'─'*72}")
    print(f"  RESULTS")
    print(f"  {'─'*72}")
    test_indices = [0, 1, 4, 7, 9]
    for i in test_indices:
        article = articles[i]
        expected = summaries[i]
        generated, enc_attn, cross_attn = model.summarize(article)
        ratio = len(generated.split()) / len(article.split())
        bar = "█" * int(ratio * 30) + "░" * (30 - int(ratio * 30))
        print(f"\n  Article ({len(article.split())}w): {article[:80]}...")
        print(f"  Expected:  {expected}")
        print(f"  Generated: {generated}")
        print(f"  Compression: [{bar}] {ratio:.0%}")

    # Cross-attention heatmap for first article
    print(f"\n  {'─'*72}")
    print(f"  CROSS-ATTENTION MAP (How decoder attends to source when generating summary)")
    print(f"  {'─'*72}\n")
    summary, enc_attn, cross_attn = model.summarize(articles[0])
    src_tokens = re.findall(r"\b\w+\b", articles[0].lower())
    sum_tokens = summary.split()
    last_cross = cross_attn[-1][0].mean(dim=0)

    print(f"  {'Summary Word':<18} → Top Attended Source Words")
    print(f"  {'─'*18}   {'─'*52}")
    for t_idx, t_word in enumerate(sum_tokens[:12]):
        if t_idx + 1 >= last_cross.size(0):
            break
        attn_row = last_cross[t_idx + 1]
        topk = attn_row.topk(min(4, len(src_tokens)))
        attended = []
        for k_idx, k_score in zip(topk.indices, topk.values):
            k_i = k_idx.item() - 1  # offset for SOS
            if 0 <= k_i < len(src_tokens):
                attended.append(f"{src_tokens[k_i]}({k_score.item():.2f})")
        print(f"  {t_word:<18} → {', '.join(attended)}")

    return model


# ============================================================================
# APPROACH 2: EXTRACTIVE SUMMARIZATION
# ============================================================================

def run_extractive():
    print("\n" + "=" * 76)
    print("  APPROACH 2: EXTRACTIVE SUMMARIZATION (Attention-Based Sentence Ranking)")
    print("=" * 76)
    print("  SELECTS the most important sentences — no text generation.")

    DOCUMENTS = {
        "AI Progress": (
            "Artificial intelligence has made remarkable progress in recent years. "
            "Large language models can now write code and answer complex questions. "
            "The transformer architecture introduced in 2017 revolutionized the field. "
            "Self attention allows models to process all tokens in parallel. "
            "Training these models requires enormous computational resources. "
            "Companies are investing billions of dollars in AI research. "
            "Concerns about AI safety and alignment are growing rapidly. "
            "Researchers are developing techniques to make AI systems more reliable. "
            "The economic impact of AI could reach trillions of dollars annually. "
            "AI is being applied in healthcare finance education and manufacturing."
        ),
        "Climate Action": (
            "Global temperatures have risen more than one degree celsius since preindustrial times. "
            "The burning of fossil fuels is the primary driver of climate change. "
            "Extreme weather events are increasing in both frequency and intensity. "
            "Sea levels are rising threatening coastal communities worldwide. "
            "Solar and wind energy costs have dropped dramatically in the last decade. "
            "Electric vehicles are replacing traditional gasoline powered cars. "
            "Carbon capture technology shows promise for reducing industrial emissions. "
            "International agreements aim to limit warming to one point five degrees. "
            "Young people are leading climate activism movements around the world. "
            "Sustainable agriculture practices can help reduce greenhouse gas emissions."
        ),
        "Space Exploration": (
            "NASA plans to return humans to the moon through the Artemis program. "
            "SpaceX has developed reusable rockets dramatically reducing launch costs. "
            "The James Webb Space Telescope has revealed stunning images of distant galaxies. "
            "Mars remains a primary target for human exploration in the coming decades. "
            "Private companies are making space more accessible than ever before. "
            "The International Space Station has hosted astronauts from many countries. "
            "Scientists are searching for signs of life on Mars and Europa. "
            "Space debris is becoming a serious concern for orbital operations. "
            "Satellite internet constellations are providing global connectivity. "
            "Space tourism is becoming a reality for wealthy private citizens."
        ),
    }

    all_text = " ".join(DOCUMENTS.values())
    vocab = Vocab([all_text])

    # Build a simple encoder
    class SimpleEncoder(nn.Module):
        def __init__(self, vocab_size, d_model=64, nhead=4, d_ff=128, n_layers=2):
            super().__init__()
            self.emb = nn.Embedding(vocab_size, d_model)
            self.pos = SinusoidalPE(d_model)
            self.layers = nn.ModuleList([EncoderLayer(d_model, nhead, d_ff) for _ in range(n_layers)])
            self.d_model = d_model
        def forward(self, x):
            x = self.pos(self.emb(x) * math.sqrt(self.d_model))
            all_attn = []
            for layer in self.layers:
                x, aw = layer(x)
                all_attn.append(aw)
            return x, all_attn

    encoder = SimpleEncoder(len(vocab)).to(DEVICE)
    encoder.eval()
    print(f"\n  Vocab: {len(vocab)} | Encoder params: {sum(p.numel() for p in encoder.parameters()):,}")

    def split_sentences(text):
        sents = re.split(r'(?<=[.!?])\s+', text.strip())
        return [s.strip() for s in sents if len(s.strip()) > 5]

    for doc_name, doc_text in DOCUMENTS.items():
        sentences = split_sentences(doc_text)
        print(f"\n  {'─'*72}")
        print(f"  {doc_name} — {len(sentences)} sentences, {len(doc_text.split())} words")
        print(f"  {'─'*72}")

        # Encode document
        ids = vocab.encode(doc_text, add_special=False)
        with torch.no_grad():
            _, all_attn = encoder(torch.tensor([ids]).to(DEVICE))

        # Token importance from last layer attention
        attn = all_attn[-1][0].mean(dim=0)  # (seq, seq)
        token_importance = attn.sum(dim=0)   # how much attention each token receives

        # Score sentences
        doc_tokens = re.findall(r"\b\w+\b", doc_text.lower())
        scores = []
        pos = 0
        for sent in sentences:
            n = len(re.findall(r"\b\w+\b", sent.lower()))
            if pos + n <= len(token_importance):
                sc = token_importance[pos:pos+n].mean().item()
            else:
                sc = 0.0
            scores.append(sc)
            pos += n

        # Rank
        ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
        top3 = sorted([idx for idx, _ in ranked[:3]])
        selected = [sentences[i] for i in top3]

        print(f"\n  Rankings:")
        print(f"  {'Rank':<5} {'Score':<8} {'Sentence'}")
        print(f"  {'─'*5} {'─'*8} {'─'*55}")
        for rank, (idx, sc) in enumerate(ranked, 1):
            sel = " ★" if idx in top3 else "  "
            trunc = sentences[idx][:55] + ("..." if len(sentences[idx]) > 55 else "")
            print(f"  {rank:<5} {sc:<8.4f} {trunc}{sel}")

        print(f"\n  Summary (top 3 sentences):")
        for i, s in enumerate(selected, 1):
            print(f"    [{i}] {s}")

        sw = sum(len(s.split()) for s in selected)
        tw = len(doc_text.split())
        print(f"  Compression: {sw}/{tw} words ({sw/tw:.0%})")


# ============================================================================
# MAIN
# ============================================================================

def main():
    print("=" * 76)
    print("  TEXT SUMMARIZATION WITH TRANSFORMERS — From Scratch")
    print("=" * 76)

    run_abstractive()
    run_extractive()

    print("\n" + "=" * 76)
    print("  ABSTRACTIVE vs EXTRACTIVE — Key Differences")
    print("=" * 76)
    print("""
    ┌───────────────────────────┬─────────────────────────────────────────┐
    │  ABSTRACTIVE              │  EXTRACTIVE                           │
    │  (Encoder-Decoder)        │  (Encoder + Ranking)                  │
    ├───────────────────────────┼─────────────────────────────────────────┤
    │  Generates NEW text       │  Selects EXISTING sentences            │
    │  Can paraphrase & merge   │  Preserves original wording            │
    │  Needs paired training    │  Works with attention alone            │
    │  Risk of hallucination    │  No hallucination risk                 │
    │  Cross-attention is key   │  Self-attention scores importance      │
    │  Slower (autoregressive)  │  Faster (single forward pass)          │
    └───────────────────────────┴─────────────────────────────────────────┘
    """)


if __name__ == "__main__":
    main()


Device: cpu
  TEXT SUMMARIZATION WITH TRANSFORMERS — From Scratch

  APPROACH 1: ABSTRACTIVE SUMMARIZATION (Encoder-Decoder Transformer)
  The model GENERATES new summary text word by word.

  Source vocab: 286 | Target vocab: 120 | Pairs: 12
  Parameters: 729,976

  Training 150 epochs...
    Epoch   1  Loss: 5.0023
    Epoch  30  Loss: 1.9435
    Epoch  60  Loss: 0.4577
    Epoch  90  Loss: 0.1121
    Epoch 120  Loss: 0.0598
    Epoch 150  Loss: 0.0392
  Done in 17.1s

  ────────────────────────────────────────────────────────────────────────
  RESULTS
  ────────────────────────────────────────────────────────────────────────

  Article (35w): artificial intelligence has transformed many industries including healthcare fin...
  Expected:  ai transforms industries through large language models trained on big data
  Generated: ai transforms industries through large language models trained on big data
  Compression: [█████████░░░░░░░░░░░░░░░░░░░░░] 31%

  Article (37w): the global clima

In [ ]:
Loss
  ↑
  └── Output Projection (gradients flow back)
        ↑
        └── Decoder Layers (each layer gets gradients)
              ↑
              └── Cross-Attention (gradients to encoder)
                    ↑
                    └── Encoder Layers
                          ↑
                          └── Embeddings

In [ ]:
┌─────────────────────────────────────────────────────────────────────┐
│                    TEXT SUMMARIZATION SYSTEM                         │
├──────────────────────────────┬──────────────────────────────────────┤
│   ABSTRACTIVE PATH           │        EXTRACTIVE PATH               │
│   (Generates new text)       │        (Selects sentences)           │
├──────────────────────────────┼──────────────────────────────────────┤
│                               │                                      │
│  Input: Article              │  Input: Document                     │
│       ↓                       │       ↓                              │
│  [Tokenizer] → IDs           │  [Tokenizer] → IDs                   │
│       ↓                       │       ↓                              │
│  [Encoder] → Representations │  [Encoder] → Attention Weights       │
│       ↓                       │       ↓                              │
│  [Decoder] → Generates       │  [Importance Scoring]                │
│       ↓                       │       ↓                              │
│  [Output] → Summary          │  [Sentence Ranking] → Summary        │
│                               │                                      │
└──────────────────────────────┴──────────────────────────────────────┘

In [ ]:
Input x: (batch, 40, 128)
    ↓
Self-Attention: (batch, 40, 128) [Each token looks at all tokens]
    ↓
Add + LayerNorm: (batch, 40, 128)
    ↓
Feed-Forward: (batch, 40, 128) [Each token processed independently]
    ↓
Add + LayerNorm: (batch, 40, 128)
    ↓
Output: (batch, 40, 128) [Same shape, transformed content]

In [ ]:
┌─────────────────────────────────────────────────────────────────┐
│                        ENCODER STACK                            │
├─────────────────────────────────────────────────────────────────┤
│                                                                  │
│  src_ids: [1, 45, 23, 67, 89, 2]                               │
│      ↓                                                          │
│  [Embedding] → (6, 128)                                         │
│      ↓                                                          │
│  [Positional Encoding] → (6, 128)                              │
│      ↓                                                          │
│  ┌──────────────────────────────────────┐                      │
│  │      EncoderLayer 1                  │                      │
│  │  ┌────────────────────────────────┐ │                      │
│  │  │ Self-Attention (no mask)       │ │                      │
│  │  │ Each token → All tokens        │ │                      │
│  │  │ "healthcare" attends to "AI"   │ │                      │
│  │  └────────────────────────────────┘ │                      │
│  │  ↓ + Residual + LayerNorm           │                      │
│  │  ┌────────────────────────────────┐ │                      │
│  │  │ Feed-Forward                   │ │                      │
│  │  │ Processes each position        │ │                      │
│  │  └────────────────────────────────┘ │                      │
│  │  ↓ + Residual + LayerNorm           │                      │
│  └──────────────────────────────────────┘                      │
│      ↓                                                          │
│  ┌──────────────────────────────────────┐                      │
│  │      EncoderLayer 2                  │                      │
│  │  (Same structure, deeper features)   │                      │
│  └──────────────────────────────────────┘                      │
│      ↓                                                          │
│  enc_out: (6, 128) - Rich representations of each source token │
└─────────────────────────────────────────────────────────────────┘
                              ↓
┌─────────────────────────────────────────────────────────────────┐
│                        DECODER STACK                            │
├─────────────────────────────────────────────────────────────────┤
│                                                                  │
│  tgt_in: [1, 45, 67, 89]  (SOS, AI, transforms, healthcare)   │
│      ↓                                                          │
│  [Embedding] → (4, 128)                                         │
│      ↓                                                          │
│  [Positional Encoding] → (4, 128)                              │
│      ↓                                                          │
│  ┌──────────────────────────────────────┐                      │
│  │      DecoderLayer 1                  │                      │
│  │  ┌────────────────────────────────┐ │                      │
│  │  │ Masked Self-Attention          │ │                      │
│  │  │ Token 0 sees: [SOS]            │ │                      │
│  │  │ Token 1 sees: [SOS, AI]        │ │                      │
│  │  │ Token 2 sees: [SOS, AI, trans] │ │                      │
│  │  │ Token 3 sees: [SOS, AI, trans, healthcare]             │
│  │  └────────────────────────────────┘ │                      │
│  │  ↓ + Residual + LayerNorm           │                      │
│  │  ┌────────────────────────────────┐ │                      │
│  │  │ Cross-Attention                │ │                      │
│  │  │ Query from decoder             │ │                      │
│  │  │ Key/Value from encoder_out     │ │                      │
│  │  │ "AI" attends to source "AI"    │ │                      │
│  │  └────────────────────────────────┘ │                      │
│  │  ↓ + Residual + LayerNorm           │                      │
│  │  ┌────────────────────────────────┐ │                      │
│  │  │ Feed-Forward                   │ │                      │
│  │  └────────────────────────────────┘ │                      │
│  │  ↓ + Residual + LayerNorm           │                      │
│  └──────────────────────────────────────┘                      │
│      ↓                                                          │
│  ┌──────────────────────────────────────┐                      │
│  │      DecoderLayer 2                  │                      │
│  │  (Same structure, deeper processing) │                      │
│  └──────────────────────────────────────┘                      │
│      ↓                                                          │
│  dec_out: (4, 128)                                             │
│      ↓                                                          │
│  [Output Projection] → (4, tgt_vocab_size)                     │
│      ↓                                                          │
│  logits: (4, 50) - Raw scores for each target word            │
└─────────────────────────────────────────────────────────────────┘

In [ ]:
                    AbstractiveSummarizer
                           │
            ┌──────────────┼──────────────┐
            │              │              │
        Encoder         Decoder     OutputProjection
           │               │              │
           │               │         (Linear layer)
    ┌──────┼──────┐   ┌────┼────┐
    │      │      │   │    │    │
Embedding  Pos  N×Encoder  Pos  N×Decoder
           Enc  Layers     Enc  Layers
                        │
                ┌───────┼───────┐
                │       │       │
          Self-Atten  Cross-Atten  FFN
                │       │
                └───────┘
                    │
            MultiHeadAttention
                    │
        ┌───────────┼───────────┐
        │           │           │
      W_q, W_k, W_v, W_o + softmax + dropout

In [ ]:
# Your prompt: "Explain quantum computing simply"
# GPT-5 uses Byte Pair Encoding (BPE) tokenizer

input_text = "Explain quantum computing simply"

# Tokenization process:
tokens = tokenizer.encode(input_text)
# Result: [24389, 8923, 456, 11234, 5678, 1234]
# Each token is ~3-4 characters

# Visual breakdown:
"Explain"  → token 24389
" quantum" → token 8923  (space + word)
" comput"  → token 456   (subword!)
"ing"      → token 11234
" simply"  → token 5678
"?"        → token 1234 (if you added it)

# Special tokens added automatically:
# [BOS] = Beginning of sequence (token 1)
# [EOS] = End of sequence (token 2)
final_tokens = [1] + tokens + [2]
# Result: [1, 24389, 8923, 456, 11234, 5678, 1234, 2]

In [ ]:
# Each token ID maps to a high-dimensional vector
token_embeddings = token_embedding(token_ids)
# Input shape: (8,) → Output shape: (8, 12288)

# Visual example (simplified to 10 dimensions instead of 12288):
token_id 1    → [0.12, -0.34, 0.56, 0.78, -0.21, ...]  # BOS token
token_id 24389 → [0.45, 0.23, -0.67, 0.89, 0.11, ...]  # "Explain"
token_id 8923  → [0.34, -0.12, 0.78, -0.45, 0.67, ...]  # "quantum"
token_id 456   → [0.67, 0.45, -0.23, 0.12, -0.78, ...]  # "comput"
# ... and so on

In [ ]:
# GPT uses LEARNED position embeddings (not sinusoidal)
position_ids = [0, 1, 2, 3, 4, 5, 6, 7]
position_embeddings = position_embedding(position_ids)
# Shape: (8, 12288)

# Position 0 embedding: [0.01, 0.02, -0.01, ...]  # BOS position
# Position 1 embedding: [0.02, 0.03, -0.02, ...]  # "Explain" position
# Position 2 embedding: [0.03, 0.04, -0.03, ...]  # "quantum" position

In [ ]:
# Simple addition
input_embeddings = token_embeddings + position_embeddings
# Shape: (8, 12288)

# This is the final input to the first decoder block
# Each token is now a rich vector representing its meaning AND position

In [ ]:
Your Prompt: "Explain quantum computing simply"
    ↓
[0.0ms] Tokenization → [1, 24389, 8923, 456, 11234, 5678, 1234, 2]
    ↓
[0.1ms] Embedding → (8, 12288) vectors
    ↓
[50ms] 120 Decoder Layers (each with attention + FFN)
    ↓
[1ms] Language Modeling Head → Probabilities over 100K tokens
    ↓
[5ms] Sampling → "Quantum" (token 3456)
    ↓
[50ms] Process new token through 120 layers (with KV-cache)
    ↓
[5ms] Sample next → "computing" (token 7890)
    ↓
[repeat 100-500 times until EOS]
    ↓
[1ms] Detokenization → Human-readable response
    ↓
Total: ~2-10 seconds for 500-token response

The entire time, your prompt is being transformed through
billions of mathematical operations across thousands of GPUs,
orchestrated by one of the most complex software systems ever built!

In [3]:
"""
=============================================================================
 QUESTION ANSWERING USING TRANSFORMERS — From Scratch (No Downloads)
=============================================================================
 Two complete QA approaches, fully self-contained:
   1. EXTRACTIVE QA — Transformer predicts start/end positions of the
      answer span within the context (like SQuAD / BERT-QA)
   2. GENERATIVE QA — Encoder-Decoder Transformer generates the answer
      word by word (like T5 / GPT for QA)

 Everything is built from raw PyTorch — no HuggingFace, no pre-trained models.
=============================================================================
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import re
from collections import Counter

torch.manual_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ============================================================================
# SHARED: Transformer Building Blocks
# ============================================================================

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, nhead, dropout=0.1):
        super().__init__()
        assert d_model % nhead == 0
        self.d_k, self.nhead = d_model // nhead, nhead
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        B = q.size(0)
        Q = self.W_q(q).view(B, -1, self.nhead, self.d_k).transpose(1, 2)
        K = self.W_k(k).view(B, -1, self.nhead, self.d_k).transpose(1, 2)
        V = self.W_v(v).view(B, -1, self.nhead, self.d_k).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        w = self.dropout(F.softmax(scores, dim=-1))
        out = torch.matmul(w, V).transpose(1, 2).contiguous().view(B, -1, self.nhead * self.d_k)
        return self.W_o(out), w


class FFN(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(),
                                 nn.Dropout(dropout), nn.Linear(d_ff, d_model))
    def forward(self, x): return self.net(x)


class PosEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x): return self.dropout(x + self.pe[:, :x.size(1)])


class Vocab:
    SPECIALS = ["<PAD>", "<SOS>", "<EOS>", "<UNK>", "<SEP>"]
    def __init__(self, texts, min_freq=1):
        c = Counter()
        for t in texts:
            c.update(self._tok(t))
        self.itos = list(self.SPECIALS) + [w for w, n in c.items() if n >= min_freq]
        self.stoi = {w: i for i, w in enumerate(self.itos)}
    def _tok(self, t): return re.findall(r"\b\w+\b", t.lower())
    def encode(self, t, special=True):
        ids = [self.stoi.get(w, self.stoi["<UNK>"]) for w in self._tok(t)]
        return ([self.stoi["<SOS>"]] + ids + [self.stoi["<EOS>"]]) if special else ids
    def decode(self, ids):
        skip = {"<PAD>", "<SOS>", "<EOS>", "<SEP>"}
        return " ".join(self.itos[i] for i in ids if self.itos[i] not in skip)
    @property
    def pad(self): return self.stoi["<PAD>"]
    @property
    def sep(self): return self.stoi["<SEP>"]
    def __len__(self): return len(self.itos)


# ============================================================================
# QA TRAINING DATA
# ============================================================================

QA_DATA = [
    # (context, question, answer_text)
    ("Python was created by Guido van Rossum and first released in 1991. "
     "It is a high level general purpose programming language.",
     "Who created Python?", "guido van rossum"),

    ("Python was created by Guido van Rossum and first released in 1991. "
     "It is a high level general purpose programming language.",
     "When was Python first released?", "1991"),

    ("The capital of France is Paris. France is located in Western Europe. "
     "Paris is known for the Eiffel Tower which was built in 1889.",
     "What is the capital of France?", "paris"),

    ("The capital of France is Paris. France is located in Western Europe. "
     "Paris is known for the Eiffel Tower which was built in 1889.",
     "When was the Eiffel Tower built?", "1889"),

    ("The capital of France is Paris. France is located in Western Europe. "
     "Paris is known for the Eiffel Tower which was built in 1889.",
     "Where is France located?", "western europe"),

    ("Alexander Graham Bell invented the telephone in 1876. "
     "He was born in Edinburgh Scotland and later moved to Canada and the United States.",
     "Who invented the telephone?", "alexander graham bell"),

    ("Alexander Graham Bell invented the telephone in 1876. "
     "He was born in Edinburgh Scotland and later moved to Canada and the United States.",
     "When was the telephone invented?", "1876"),

    ("Alexander Graham Bell invented the telephone in 1876. "
     "He was born in Edinburgh Scotland and later moved to Canada and the United States.",
     "Where was Bell born?", "edinburgh scotland"),

    ("The transformer architecture was introduced in 2017 by Vaswani and colleagues. "
     "It uses self attention to process sequences in parallel. "
     "The original model had 6 layers and 8 attention heads with a dimension of 512.",
     "When was the transformer introduced?", "2017"),

    ("The transformer architecture was introduced in 2017 by Vaswani and colleagues. "
     "It uses self attention to process sequences in parallel. "
     "The original model had 6 layers and 8 attention heads with a dimension of 512.",
     "How many layers did the original transformer have?", "6"),

    ("The transformer architecture was introduced in 2017 by Vaswani and colleagues. "
     "It uses self attention to process sequences in parallel. "
     "The original model had 6 layers and 8 attention heads with a dimension of 512.",
     "How many attention heads did it use?", "8"),

    ("DNA stands for deoxyribonucleic acid. It carries genetic instructions "
     "for the development of all known living organisms and many viruses.",
     "What does DNA stand for?", "deoxyribonucleic acid"),

    ("DNA stands for deoxyribonucleic acid. It carries genetic instructions "
     "for the development of all known living organisms and many viruses.",
     "What does DNA carry?", "genetic instructions"),

    ("Our solar system has eight planets. The four inner planets Mercury Venus "
     "Earth and Mars are rocky. Jupiter Saturn Uranus and Neptune are gas giants.",
     "How many planets are in our solar system?", "eight"),

    ("Our solar system has eight planets. The four inner planets Mercury Venus "
     "Earth and Mars are rocky. Jupiter Saturn Uranus and Neptune are gas giants.",
     "Which planets are rocky?", "mercury venus earth and mars"),

    ("Singapore is a city state in Southeast Asia. It has a population of "
     "about 5.9 million people. The country was founded in 1965 by Lee Kuan Yew.",
     "What is the population of Singapore?", "5 9 million"),

    ("Singapore is a city state in Southeast Asia. It has a population of "
     "about 5.9 million people. The country was founded in 1965 by Lee Kuan Yew.",
     "Who founded Singapore?", "lee kuan yew"),

    ("Singapore is a city state in Southeast Asia. It has a population of "
     "about 5.9 million people. The country was founded in 1965 by Lee Kuan Yew.",
     "When was Singapore founded?", "1965"),

    ("Machine learning was defined by Arthur Samuel in 1959 as the ability of "
     "computers to learn without being explicitly programmed. Deep learning uses "
     "neural networks with many layers to learn complex patterns from data.",
     "Who defined machine learning?", "arthur samuel"),

    ("Machine learning was defined by Arthur Samuel in 1959 as the ability of "
     "computers to learn without being explicitly programmed. Deep learning uses "
     "neural networks with many layers to learn complex patterns from data.",
     "When was machine learning defined?", "1959"),
]


# ============================================================================
# APPROACH 1: EXTRACTIVE QA (Predict Start/End Span)
# ============================================================================
# Input format:  [question tokens] <SEP> [context tokens]
# Model outputs: start_logits (one per token) + end_logits (one per token)
# Answer:        text from argmax(start) to argmax(end) in the context portion
#
# This is exactly how BERT-QA / SQuAD models work.
# ============================================================================

class ExtractiveQAModel(nn.Module):
    """
    Encoder-only Transformer for extractive QA.
    Jointly encodes [question <SEP> context] and predicts answer span boundaries.
    """
    def __init__(self, vocab_size, d_model=128, nhead=4, d_ff=256, n_layers=3, dropout=0.1):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model)
        self.segment_emb = nn.Embedding(2, d_model)  # 0=question, 1=context
        self.pos = PosEncoding(d_model, dropout=dropout)
        self.d_model = d_model

        self.layers = nn.ModuleList()
        for _ in range(n_layers):
            self.layers.append(nn.ModuleDict({
                'attn': MultiHeadAttention(d_model, nhead, dropout),
                'ffn': FFN(d_model, d_ff, dropout),
                'norm1': nn.LayerNorm(d_model),
                'norm2': nn.LayerNorm(d_model),
                'drop1': nn.Dropout(dropout),
                'drop2': nn.Dropout(dropout),
            }))

        # QA heads — predict which token is start / end of answer
        self.start_head = nn.Linear(d_model, 1)
        self.end_head = nn.Linear(d_model, 1)

    def forward(self, input_ids, segment_ids, attention_mask=None):
        # Embed: token embedding + segment embedding + positional encoding
        x = self.emb(input_ids) * math.sqrt(self.d_model)
        x = x + self.segment_emb(segment_ids)
        x = self.pos(x)

        # Mask for padding
        mask = None
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(1).unsqueeze(2)

        # Encoder layers
        all_attn = []
        for layer in self.layers:
            attn_out, attn_w = layer['attn'](x, x, x, mask)
            x = layer['norm1'](x + layer['drop1'](attn_out))
            x = layer['norm2'](x + layer['drop2'](layer['ffn'](x)))
            all_attn.append(attn_w)

        # QA predictions
        start_logits = self.start_head(x).squeeze(-1)  # (B, seq_len)
        end_logits = self.end_head(x).squeeze(-1)       # (B, seq_len)

        return start_logits, end_logits, all_attn


def run_extractive_qa():
    print("\n" + "=" * 76)
    print("  APPROACH 1: EXTRACTIVE QA (Start/End Span Prediction)")
    print("=" * 76)
    print("  Model finds WHERE in the context the answer is located.")
    print("  Input: [question <SEP> context]  →  Output: start_pos, end_pos")

    # Build vocab from all questions + contexts + answers
    all_texts = []
    for ctx, q, a in QA_DATA:
        all_texts.extend([ctx, q, a])
    vocab = Vocab(all_texts)
    print(f"\n  Vocab: {len(vocab)} tokens | Training examples: {len(QA_DATA)}")

    # Prepare data: [question_tokens, <SEP>, context_tokens]
    def prepare_extractive(data, vocab, max_len=120):
        all_input_ids, all_segment_ids, all_masks = [], [], []
        all_start_pos, all_end_pos = [], []
        valid_indices = []

        for idx, (ctx, question, answer) in enumerate(data):
            q_tokens = vocab._tok(question)
            c_tokens = vocab._tok(ctx)
            a_tokens = vocab._tok(answer)

            # Build: [q_tokens] <SEP> [c_tokens]
            combined = q_tokens + ["<SEP>"] + c_tokens
            sep_pos = len(q_tokens)

            # Encode
            ids = [vocab.stoi.get(w, vocab.stoi["<UNK>"]) for w in combined]
            segments = [0] * (sep_pos + 1) + [1] * len(c_tokens)  # 0=question, 1=context

            # Find answer span in context tokens
            start, end = None, None
            for i in range(len(c_tokens) - len(a_tokens) + 1):
                if c_tokens[i:i+len(a_tokens)] == a_tokens:
                    start = sep_pos + 1 + i  # offset by question + SEP
                    end = start + len(a_tokens) - 1
                    break

            if start is None:
                continue  # skip if answer not found

            # Pad to max_len
            pad_len = max_len - len(ids)
            if pad_len < 0:
                continue  # skip too long

            ids += [vocab.pad] * pad_len
            segments += [1] * pad_len
            mask = [1] * (max_len - pad_len) + [0] * pad_len

            all_input_ids.append(ids)
            all_segment_ids.append(segments)
            all_masks.append(mask)
            all_start_pos.append(start)
            all_end_pos.append(end)
            valid_indices.append(idx)

        return (torch.tensor(all_input_ids),
                torch.tensor(all_segment_ids),
                torch.tensor(all_masks),
                torch.tensor(all_start_pos),
                torch.tensor(all_end_pos),
                valid_indices)

    input_ids, seg_ids, masks, starts, ends, valid_idx = prepare_extractive(QA_DATA, vocab)
    input_ids = input_ids.to(DEVICE)
    seg_ids = seg_ids.to(DEVICE)
    masks = masks.to(DEVICE)
    starts = starts.to(DEVICE)
    ends = ends.to(DEVICE)

    print(f"  Successfully prepared: {len(valid_idx)} examples")

    # Show a tokenized example
    print(f"\n  Example tokenization:")
    ex_tokens = [vocab.itos[i] if i < len(vocab.itos) else "?" for i in input_ids[0].tolist()]
    sep_idx = ex_tokens.index("<SEP>") if "<SEP>" in ex_tokens else -1
    print(f"    Question: {' '.join(ex_tokens[:sep_idx])}")
    print(f"    Context:  {' '.join(t for t in ex_tokens[sep_idx+1:] if t != '<PAD>')}")
    print(f"    Answer span: [{starts[0].item()}, {ends[0].item()}] = "
          f"'{' '.join(ex_tokens[starts[0].item():ends[0].item()+1])}'")

    # Build and train model
    model = ExtractiveQAModel(len(vocab)).to(DEVICE)
    params = sum(p.numel() for p in model.parameters())
    print(f"\n  Model: d=128, heads=4, layers=3, params={params:,}")

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    EPOCHS = 250
    print(f"  Training {EPOCHS} epochs...")
    t0 = time.time()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        s_logits, e_logits, _ = model(input_ids, seg_ids, masks)

        loss = criterion(s_logits, starts) + criterion(e_logits, ends)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        if epoch % 50 == 0 or epoch == 1:
            pred_s = s_logits.argmax(-1)
            pred_e = e_logits.argmax(-1)
            exact = ((pred_s == starts) & (pred_e == ends)).float().mean().item()
            print(f"    Epoch {epoch:3d}  Loss: {loss.item():.4f}  Exact Match: {exact:.2%}")

    print(f"  Done in {time.time()-t0:.1f}s")

    # Inference
    print(f"\n  {'─'*72}")
    print(f"  EXTRACTIVE QA RESULTS")
    print(f"  {'─'*72}")

    model.eval()
    with torch.no_grad():
        s_logits, e_logits, all_attn = model(input_ids, seg_ids, masks)

    correct = 0
    for i, data_idx in enumerate(valid_idx):
        ctx, question, expected = QA_DATA[data_idx]
        tokens = [vocab.itos[t] if t < len(vocab.itos) else "?" for t in input_ids[i].tolist()]
        sep_idx = tokens.index("<SEP>") if "<SEP>" in tokens else 0

        # Only consider context tokens for prediction
        sl = s_logits[i].clone()
        el = e_logits[i].clone()
        sl[:sep_idx+1] = float("-inf")  # mask out question tokens
        el[:sep_idx+1] = float("-inf")

        pred_start = sl.argmax().item()
        pred_end = el.argmax().item()
        if pred_end < pred_start:
            pred_end = pred_start

        predicted = " ".join(tokens[pred_start:pred_end+1])

        # Confidence
        s_prob = F.softmax(sl, -1)[pred_start].item()
        e_prob = F.softmax(el, -1)[pred_end].item()
        conf = s_prob * e_prob

        match = predicted.strip() == expected.strip()
        if match:
            correct += 1
        status = "✓" if match else "✗"
        bar = "█" * int(conf * 20) + "░" * (20 - int(conf * 20))

        print(f"\n  {status} Q: {question}")
        print(f"    Expected: '{expected}'")
        print(f"    Got:      '{predicted}' (span [{pred_start},{pred_end}])")
        print(f"    Confidence: [{bar}] {conf:.4f}")

    print(f"\n  Accuracy: {correct}/{len(valid_idx)} ({correct/len(valid_idx):.1%})")

    # Visualize start/end logits for one example
    print(f"\n  {'─'*72}")
    print(f"  TOKEN-LEVEL START/END PROBABILITIES (Example 1)")
    print(f"  {'─'*72}\n")

    ex = 0
    tokens = [vocab.itos[t] if t < len(vocab.itos) else "?" for t in input_ids[ex].tolist()]
    sep_idx = tokens.index("<SEP>") if "<SEP>" in tokens else 0
    sl = s_logits[ex].clone()
    el = e_logits[ex].clone()
    sl[:sep_idx+1] = float("-inf")
    el[:sep_idx+1] = float("-inf")
    sp = F.softmax(sl, -1)
    ep = F.softmax(el, -1)

    pred_s = sl.argmax().item()
    pred_e = el.argmax().item()

    print(f"  {'Pos':<5} {'Token':<18} {'P(start)':<10} {'P(end)':<10} {'Visual'}")
    print(f"  {'─'*5} {'─'*18} {'─'*10} {'─'*10} {'─'*25}")
    for j in range(sep_idx+1, min(sep_idx+20, len(tokens))):
        if tokens[j] == "<PAD>":
            break
        s_val, e_val = sp[j].item(), ep[j].item()
        bar_s = "▓" * int(s_val * 30)
        bar_e = "░" * int(e_val * 30)
        marker = " ◄ ANS" if pred_s <= j <= pred_e else ""
        print(f"  {j:<5} {tokens[j]:<18} {s_val:<10.4f} {e_val:<10.4f} {bar_s}{bar_e}{marker}")


# ============================================================================
# APPROACH 2: GENERATIVE QA (Encoder-Decoder, Generate Answer)
# ============================================================================
# Instead of pointing to a span, this model GENERATES the answer token by token.
# Input: Encoder receives [question <SEP> context]
# Output: Decoder generates [answer tokens]
#
# This is how T5, GPT, and modern LLMs approach QA.
# ============================================================================

class GenerativeQAModel(nn.Module):
    """Encoder-Decoder Transformer that generates answers."""
    def __init__(self, vocab_size, d_model=128, nhead=4, d_ff=256, n_enc=2, n_dec=2, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.emb = nn.Embedding(vocab_size, d_model)
        self.pos = PosEncoding(d_model, dropout=dropout)

        # Encoder
        self.enc_layers = nn.ModuleList()
        for _ in range(n_enc):
            self.enc_layers.append(nn.ModuleDict({
                'attn': MultiHeadAttention(d_model, nhead, dropout),
                'ffn': FFN(d_model, d_ff, dropout),
                'n1': nn.LayerNorm(d_model), 'n2': nn.LayerNorm(d_model),
                'd1': nn.Dropout(dropout), 'd2': nn.Dropout(dropout),
            }))

        # Decoder
        self.dec_layers = nn.ModuleList()
        for _ in range(n_dec):
            self.dec_layers.append(nn.ModuleDict({
                'self_attn': MultiHeadAttention(d_model, nhead, dropout),
                'cross_attn': MultiHeadAttention(d_model, nhead, dropout),
                'ffn': FFN(d_model, d_ff, dropout),
                'n1': nn.LayerNorm(d_model), 'n2': nn.LayerNorm(d_model),
                'n3': nn.LayerNorm(d_model),
                'd1': nn.Dropout(dropout), 'd2': nn.Dropout(dropout),
                'd3': nn.Dropout(dropout),
            }))

        self.out_proj = nn.Linear(d_model, vocab_size)
        self.vocab_size = vocab_size

    def encode(self, src, mask=None):
        x = self.pos(self.emb(src) * math.sqrt(self.d_model))
        for layer in self.enc_layers:
            a, _ = layer['attn'](x, x, x, mask)
            x = layer['n1'](x + layer['d1'](a))
            x = layer['n2'](x + layer['d2'](layer['ffn'](x)))
        return x

    def decode(self, tgt, enc_out, src_mask=None, tgt_mask=None):
        x = self.pos(self.emb(tgt) * math.sqrt(self.d_model))
        cross_attns = []
        for layer in self.dec_layers:
            a, _ = layer['self_attn'](x, x, x, tgt_mask)
            x = layer['n1'](x + layer['d1'](a))
            a, cw = layer['cross_attn'](x, enc_out, enc_out, src_mask)
            x = layer['n2'](x + layer['d2'](a))
            x = layer['n3'](x + layer['d3'](layer['ffn'](x)))
            cross_attns.append(cw)
        return self.out_proj(x), cross_attns

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        enc_out = self.encode(src, src_mask)
        return self.decode(tgt, enc_out, src_mask, tgt_mask)

    @torch.no_grad()
    def answer(self, input_ids, vocab, max_len=20):
        self.eval()
        src = input_ids.unsqueeze(0).to(DEVICE)
        src_mask = (src != vocab.pad).unsqueeze(1).unsqueeze(2)
        enc_out = self.encode(src, src_mask)

        tgt_ids = [vocab.stoi["<SOS>"]]
        for _ in range(max_len):
            tgt = torch.tensor([tgt_ids]).to(DEVICE)
            T = tgt.size(1)
            tgt_mask = torch.tril(torch.ones(T, T, device=DEVICE)).bool().unsqueeze(0).unsqueeze(0)
            logits, cross_attn = self.decode(tgt, enc_out, src_mask, tgt_mask)
            next_tok = logits[0, -1].argmax().item()
            tgt_ids.append(next_tok)
            if next_tok == vocab.stoi["<EOS>"]:
                break
        return vocab.decode(tgt_ids), cross_attn


def run_generative_qa():
    print("\n" + "=" * 76)
    print("  APPROACH 2: GENERATIVE QA (Encoder-Decoder Answer Generation)")
    print("=" * 76)
    print("  Model GENERATES the answer word by word (like T5/GPT).")
    print("  Input: encoder reads [question <SEP> context]")
    print("  Output: decoder generates [answer]")

    # Build vocab
    all_texts = []
    for ctx, q, a in QA_DATA:
        all_texts.extend([ctx, q, a])
    vocab = Vocab(all_texts)
    print(f"\n  Vocab: {len(vocab)} | Examples: {len(QA_DATA)}")

    # Prepare: source = [question <SEP> context], target = [answer]
    def prepare_generative(data, vocab, max_src=100, max_tgt=20):
        srcs, tgts = [], []
        for ctx, q, a in data:
            q_ids = [vocab.stoi.get(w, vocab.stoi["<UNK>"]) for w in vocab._tok(q)]
            c_ids = [vocab.stoi.get(w, vocab.stoi["<UNK>"]) for w in vocab._tok(ctx)]
            src = [vocab.stoi["<SOS>"]] + q_ids + [vocab.sep] + c_ids + [vocab.stoi["<EOS>"]]
            tgt = vocab.encode(a)

            # Pad
            src = src[:max_src] + [vocab.pad] * max(0, max_src - len(src))
            tgt = tgt[:max_tgt] + [vocab.pad] * max(0, max_tgt - len(tgt))
            srcs.append(src)
            tgts.append(tgt)
        return torch.tensor(srcs), torch.tensor(tgts)

    src_batch, tgt_batch = prepare_generative(QA_DATA, vocab)
    src_batch = src_batch.to(DEVICE)
    tgt_batch = tgt_batch.to(DEVICE)

    # Build model
    model = GenerativeQAModel(len(vocab)).to(DEVICE)
    params = sum(p.numel() for p in model.parameters())
    print(f"  Model: d=128, heads=4, enc=2, dec=2, params={params:,}")

    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
    criterion = nn.CrossEntropyLoss(ignore_index=vocab.pad)

    EPOCHS = 200
    print(f"  Training {EPOCHS} epochs...")
    t0 = time.time()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        tgt_in = tgt_batch[:, :-1]
        tgt_out = tgt_batch[:, 1:]

        src_mask = (src_batch != vocab.pad).unsqueeze(1).unsqueeze(2)
        B, T = tgt_in.shape
        pad_mask = (tgt_in != vocab.pad).unsqueeze(1).unsqueeze(2)
        causal = torch.tril(torch.ones(T, T, device=DEVICE)).bool()
        tgt_mask = pad_mask & causal

        logits, _ = model(src_batch, tgt_in, src_mask, tgt_mask)
        loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        if epoch % 40 == 0 or epoch == 1:
            print(f"    Epoch {epoch:3d}  Loss: {loss.item():.4f}")

    print(f"  Done in {time.time()-t0:.1f}s")

    # Inference
    print(f"\n  {'─'*72}")
    print(f"  GENERATIVE QA RESULTS")
    print(f"  {'─'*72}")

    correct = 0
    for ctx, question, expected in QA_DATA:
        q_ids = [vocab.stoi.get(w, vocab.stoi["<UNK>"]) for w in vocab._tok(question)]
        c_ids = [vocab.stoi.get(w, vocab.stoi["<UNK>"]) for w in vocab._tok(ctx)]
        src = torch.tensor([vocab.stoi["<SOS>"]] + q_ids + [vocab.sep] + c_ids + [vocab.stoi["<EOS>"]])

        answer, cross_attn = model.answer(src, vocab)
        match = answer.strip() == expected.strip()
        if match:
            correct += 1
        status = "✓" if match else "✗"
        print(f"  {status} Q: {question}")
        print(f"    Expected:  '{expected}'")
        print(f"    Generated: '{answer}'")

    print(f"\n  Accuracy: {correct}/{len(QA_DATA)} ({correct/len(QA_DATA):.1%})")

    # Cross-attention visualization for one example
    print(f"\n  {'─'*72}")
    print(f"  CROSS-ATTENTION: How decoder attends to source when generating answer")
    print(f"  {'─'*72}\n")

    ctx, q, exp = QA_DATA[0]
    q_ids = [vocab.stoi.get(w, vocab.stoi["<UNK>"]) for w in vocab._tok(q)]
    c_ids = [vocab.stoi.get(w, vocab.stoi["<UNK>"]) for w in vocab._tok(ctx)]
    src = torch.tensor([vocab.stoi["<SOS>"]] + q_ids + [vocab.sep] + c_ids + [vocab.stoi["<EOS>"]])
    src_tokens = ["<SOS>"] + vocab._tok(q) + ["<SEP>"] + vocab._tok(ctx) + ["<EOS>"]

    answer, cross_attn = model.answer(src, vocab)
    ans_tokens = answer.split()

    # Average last-layer cross attention across heads
    ca = cross_attn[-1][0].mean(dim=0)  # (tgt_len, src_len)

    print(f"  Q: {q}")
    print(f"  A: {answer}\n")
    print(f"  {'Answer Word':<18} → Top Attended Source Words")
    print(f"  {'─'*18}   {'─'*52}")
    for t, word in enumerate(ans_tokens):
        if t + 1 >= ca.size(0):
            break
        row = ca[t + 1]
        topk = row.topk(min(4, len(src_tokens)))
        parts = []
        for ki, ks in zip(topk.indices, topk.values):
            k = ki.item()
            if k < len(src_tokens):
                parts.append(f"{src_tokens[k]}({ks.item():.2f})")
        print(f"  {word:<18} → {', '.join(parts)}")


# ============================================================================
# MAIN
# ============================================================================

def main():
    print("=" * 76)
    print("  QUESTION ANSWERING WITH TRANSFORMERS — From Scratch")
    print("=" * 76)

    run_extractive_qa()
    run_generative_qa()

    print("\n" + "=" * 76)
    print("  EXTRACTIVE vs GENERATIVE QA — Comparison")
    print("=" * 76)
    print("""
    ┌────────────────────────────┬────────────────────────────────────────┐
    │  EXTRACTIVE QA             │  GENERATIVE QA                       │
    │  (Encoder-Only + Spans)    │  (Encoder-Decoder)                   │
    ├────────────────────────────┼────────────────────────────────────────┤
    │  Points to answer IN text  │  Generates answer word by word       │
    │  Answer must exist verbatim│  Can paraphrase or synthesize        │
    │  Two outputs: start, end   │  Auto-regressive token generation    │
    │  Very fast inference       │  Slower (sequential decoding)        │
    │  Cannot say "unanswerable" │  Can generate "I don't know"         │
    │  High precision when span  │  More flexible but may hallucinate   │
    │  exists in context         │                                      │
    │                            │                                      │
    │  Models: BERT, DistilBERT, │  Models: T5, GPT, BART, LLaMA       │
    │  RoBERTa, ALBERT           │                                      │
    │                            │                                      │
    │  Core mechanism:           │  Core mechanism:                     │
    │  [Q <SEP> C] → encoder    │  Encoder reads [Q <SEP> C]           │
    │  → start_logits, end_logits│  Decoder generates answer tokens     │
    │  → extract text span       │  Cross-attention bridges Q+C → A     │
    └────────────────────────────┴────────────────────────────────────────┘
    """)


if __name__ == "__main__":
    main()


Device: cpu
  QUESTION ANSWERING WITH TRANSFORMERS — From Scratch

  APPROACH 1: EXTRACTIVE QA (Start/End Span Prediction)
  Model finds WHERE in the context the answer is located.
  Input: [question <SEP> context]  →  Output: start_pos, end_pos

  Vocab: 161 tokens | Training examples: 20
  Successfully prepared: 20 examples

  Example tokenization:
    Question: who created python
    Context:  python was created by guido van rossum and first released in 1991 it is a high level general purpose programming language
    Answer span: [8, 10] = 'guido van rossum'

  Model: d=128, heads=4, layers=3, params=418,562
  Training 250 epochs...
    Epoch   1  Loss: 9.4616  Exact Match: 0.00%
    Epoch  50  Loss: 1.3144  Exact Match: 55.00%
    Epoch 100  Loss: 0.4466  Exact Match: 80.00%
    Epoch 150  Loss: 0.1101  Exact Match: 95.00%
    Epoch 200  Loss: 0.1610  Exact Match: 95.00%
    Epoch 250  Loss: 0.0789  Exact Match: 95.00%
  Done in 112.7s

  ───────────────────────────────────────────